In [1]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

df = pd.read_csv(os.path.join(path, "HAM10000_metadata.csv"))

img_dir1 = os.path.join(path, "HAM10000_images_part_1")
img_dir2 = os.path.join(path, "HAM10000_images_part_2")

def get_path(img_id):
    p1 = os.path.join(img_dir1, img_id + ".jpg")
    p2 = os.path.join(img_dir2, img_id + ".jpg")
    return p1 if os.path.exists(p1) else p2

df['path'] = df['image_id'].apply(get_path)

# Binary labels
malignant = ['mel', 'bcc', 'akiec']
df['label'] = df['dx'].apply(lambda x: 1 if x in malignant else 0)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.


In [2]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

In [3]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [4]:
from torch.utils.data import Dataset
from PIL import Image

class SkinDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.loc[idx, 'path']).convert("RGB")
        label = self.df.loc[idx, 'label']
        return self.transform(img), label

In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    SkinDataset(train_df, train_transform),
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SkinDataset(val_df, val_transform),
    batch_size=64,
    shuffle=False
)

In [6]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SkinModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.backbone(x)

model = SkinModel().to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 178MB/s]


In [19]:
pos_weight = torch.tensor([1.3]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [20]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=5
)

In [21]:
scaler = torch.cuda.amp.GradScaler()

for epoch in range(8):
    model.train()

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        with torch.cuda.amp.autocast():
            outputs = model(imgs).squeeze()
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    print(f"Epoch {epoch+1} done")

/tmp/ipykernel_711/3024337749.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_711/3024337749.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done


In [22]:
import numpy as np

model.eval()
probs = []
y_val = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)

        outputs = torch.sigmoid(model(imgs)).cpu().numpy()
        probs.extend(outputs)
        y_val.extend(labels.numpy())

probs = np.array(probs).flatten()
y_val = np.array(y_val)

In [23]:
from sklearn.metrics import f1_score
import numpy as np

best_t, best_f1 = 0, 0

for t in np.arange(0.3, 0.7, 0.01):
    preds = (probs > t).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t, "Best F1:", best_f1)

y_pred = (probs > best_t).astype(int)

Best threshold: 0.6600000000000004 Best F1: 0.7698412698412699


In [24]:
class SmoothBCE(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return nn.BCEWithLogitsLoss()(logits, targets)

criterion = SmoothBCE()

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

y_pred = (probs > best_t).astype(int)

print("Accuracy :", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall   :", recall_score(y_val, y_pred))

Accuracy : 0.9131303045431852
Precision: 0.7972602739726027
Recall   : 0.7442455242966752
